# 🚀 CryptoFL — Scaling Experiment na GPU (Google Colab)

Roda o `scaling_experiment.py` (FedAvg vs FedProx × nº de clientes) usando **GPU**.
O código detecta CUDA sozinho (`flower_fl/client.py`), então em GPU o CIFAR/ResNet-18
cai de **~30 min/round para segundos** — e somem os OOM / *ping timeout* que travavam em CPU.

**Antes de tudo:** menu **Runtime → Change runtime type → Hardware accelerator → GPU (T4)** → *Save*.

Depois rode as células em ordem (Runtime → Run all).


## 0) Confirmar que a GPU está ativa


In [ ]:
import torch, subprocess
print('torch:', torch.__version__, '| CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\u26a0\ufe0f  SEM GPU! Runtime > Change runtime type > GPU e rode de novo.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


## 1) Clonar o repositório + instalar dependências
Repo **público** → clone direto. O torch/CUDA já vêm no Colab; só falta o Flower.


In [ ]:
import os
os.chdir('/content')
!test -d cryptoFL || git clone -b claude/loving-hypatia-f54mj0 https://github.com/Luanmantegazine/cryptoFL.git
%cd /content/cryptoFL
!pip install -q 'flwr==1.7.0' 'numpy<2.0'
print('\u2705 ambiente pronto')


## 2) Teste rápido (~1 min) — confirma que conecta e usa a GPU
2 clientes, 1 round, MNIST. Se terminar com acurácia, o pipeline está OK na GPU.


In [ ]:
!KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py --clients-list 2 --rounds 1 --aggregators fedavg --repetitions 1 --dataset mnist --model mnistnet --output-dir results/_smoke


## 3) (opcional) Sweep MNIST — rápido na GPU (~15\u201325 min)
Você já rodou localmente; mantenha aqui só se quiser reproduzir / gerar as figuras.


In [ ]:
!mkdir -p logs && KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py --clients-list 2,4,6,8,10 --rounds 3 --aggregators fedavg,fedprox --repetitions 3 --dataset mnist --model mnistnet --output-dir results/scaling_mnist 2>&1 | tee logs/scaling_mnist.log


## 4) Sweep CIFAR-10 / ResNet-18 — agora VIÁVEL na GPU 🎯
Em CPU era inviável (~30 min/round + quedas). Na GPU o sweep completo roda em ~30\u201390 min.

> Só quer o **ponto único** (mais rápido, ~5\u201310 min)? Troque por `--clients-list 2 --repetitions 1`.


In [ ]:
!mkdir -p logs && KMP_DUPLICATE_LIB_OK=TRUE python scaling_experiment.py --clients-list 2,4,6,8,10 --rounds 3 --aggregators fedavg,fedprox --repetitions 3 --dataset cifar10 --model resnet18 --alpha 0.5 --output-dir results/scaling_cifar 2>&1 | tee logs/scaling_cifar.log


## 5) Ver resultados (tabela SUMMARY + figuras)


In [ ]:
import json
from IPython.display import Image, display

def summary(path):
    s = json.load(open(path)); cfg = s['config']; r = s['results']
    print(f"== {cfg['dataset'].upper()} | {cfg['aggregators']} | N={cfg['clients_list']} | reps={cfg['repetitions']} ==")
    h = f"{'agg':<9}{'N':>4}{'mean_acc':>11}{'std_acc':>10}{'mean_time_s':>13}{'n_runs':>8}"
    print(h); print('-'*len(h))
    for n in cfg['clients_list']:
        for a in cfg['aggregators']:
            e = r.get(str(n), {}).get(a)
            if e:
                print(f"{a:<9}{n:>4}{e['mean_accuracy']:>11.4f}{e['std_accuracy']:>10.4f}{e['mean_time_s']:>13.2f}{e['n_runs']:>8}")

for tag in ['results/scaling_mnist', 'results/scaling_cifar']:
    p = f'{tag}/scaling_summary.json'
    try:
        summary(p)
        for fig in ['scaling_accuracy.png', 'scaling_time.png']:
            display(Image(f'{tag}/{fig}'))
    except FileNotFoundError:
        print(f'(sem {p} ainda)')
    print()


## 6) Salvar os resultados (a sessão do Colab é efêmera!)
**Opção A** — baixar um zip. **Opção B** (célula seguinte) — salvar no Google Drive (persiste).


In [ ]:
!zip -qr resultados.zip results logs && echo 'zip pronto'
from google.colab import files
files.download('resultados.zip')


In [ ]:
# Opção B \u2014 Google Drive (descomente para usar):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results logs '/content/drive/MyDrive/cryptoFL_results' && echo 'salvo no Drive'
